In [2]:
"""	Base model class.
"""

__author__ = 'Achin Jain'
__email__ = 'achinj@seas.upenn.edu'


import numpy as np
# from numba import njit, float64, boolean, int64, prange
# from numba.experimental import jitclass

import jax
import jax.numpy as jnp
from functools import partial

# def _get_batch(num_models, x_t, u_t):
#     x_t_batch = np.empty((num_models, x_t.shape[0]), dtype=np.float64)
#     u_t_batch = np.empty((num_models, u_t.shape[0]), dtype=np.float64)
    
#     for i in range(num_models):
#         x_t_batch[i, :] = x_t
#         u_t_batch[i, :] = u_t

#     return x_t_batch, u_t_batch

# spec = [
#     ('num_models', int64),          
#     ('history_length', int64),       
#     ('last_predicted_states', float64[:, :]), 
#     ('running_cost', float64[:]),         
#     ('cost_history', float64[:, :]),       
#     ('queue_index', int64),
#     ('dt', float64),                          
#     ('cost_weights', float64[:]),  
#     ('state_size', int64),                    
# ]
# @jitclass(spec)

@jax.jit
def get_lookback_error(last_predicted_states, x_t, running_cost, cost_history, cost_weights, queue_index):
    running_cost = running_cost - cost_history[:, queue_index]
    cost = jnp.sum(jnp.square(x_t[None, :] - last_predicted_states) * cost_weights[None, :], axis = 1)
    cost_history = cost_history.at[:, queue_index].set(cost)

    running_cost = running_cost + cost
    
    return cost_history, running_cost

@jax.jit
def find_best_model(running_cost):
    return jnp.argmin(running_cost)

class LBHistory:

    def __init__(self, num_models, history_length, dt, cost_weights, state_size, integrator, dynamics_bank, diffeq):
        self.num_models = num_models
        self.history_length = history_length
        self.last_predicted_states = jnp.zeros((self.num_models, state_size))
        self.running_cost = jnp.zeros(self.num_models)
        self.cost_history = jnp.zeros((self.num_models, self.history_length))
        self.queue_index = 0
        self.dt = dt
        self.cost_weights = cost_weights
        self.state_size = state_size
        
        self.dynamics_bank = dynamics_bank
        self.diffeq = jax.jit(diffeq)
        self.integrator = jax.jit(integrator(
            dynamics_bank.param_bank,
            # self.dynamics_bank.get_known_params()
            ))


    def predict_states(self, x_t, u_t):
        """Batched version of _integrate"""
        self.last_predicted_states = self.integrator(
            x_t,
            u_t,
            self.dt)
        
    def update_lookback_error(self, x_t):
        self.cost_history, self.running_cost = get_lookback_error(
            self.last_predicted_states,
            x_t, 
            self.running_cost,
            self.cost_history,
            self.cost_weights,
            self.queue_index
        )
        self.queue_index = (self.queue_index + 1) % self.history_length
    

    def reset(self):
        self.last_predicted_states = jnp.zeros((self.num_models, self.state_size))
        self.running_cost = jnp.zeros(self.num_models)
        self.cost_history = jnp.zeros((self.num_models, self.history_length))
        self.queue_index = 0

    def get_best_model(self):
        return find_best_model(self.running_cost)

           
# integrate vehicle dynamics by 1 step
import numpy as np
# from numba import njit, float64, boolean, int64
# from numba.experimental import jitclass

import jax
import jax.numpy as jnp
from functools import partial


# @njit(parallel=True)
# @njit(fastmath=True)

def integrator(bank_params ):
    """Returns a function that integrates with fixed bank_params."""
    
    def odeintRK4_batch(x0, u, h):
        def step(b_p, x_t):
            return diffequation(b_p,  x_t, u)

        def rk4(b_p):
            k1 = h * step(b_p, x0)
            k2 = h * step(b_p, x0 + k1 / 2)
            k3 = h * step(b_p, x0 + k2 / 2)
            k4 = h * step(b_p, x0 + k3)
            return x0 + (k1 + 2 * k2 + 2 * k3 + k4) / 6

        return jax.vmap(rk4)(bank_params)
    
    return jax.jit(odeintRK4_batch)

@jax.jit
def diffequation(
    bank_params, #known_params,
      x, u):
    """	write dynamics as first order ODE: dxdt = f(x(t))
        x is a 6x1 vector: [x, y, psi, vx, vy, omega]^T
        u is a 2x1 vector: [acc/pwm, steer]^T
    """
    g = 9.81
    steer = u[1]
    psi = x[2]
    vx = x[3]
    vy = x[4]
    omega = x[5]

    mass, Iz, lf, lr, pitch, roll = 3.74, 0.04712, 0.15875,0.17145, 0, 0
    Ffy, Frx, Fry = _calc_forces(bank_params, x, u)

    return jnp.array([
        vx*jnp.cos(psi) - vy*jnp.sin(psi),
        vx*jnp.sin(psi) + vy*jnp.cos(psi),
        omega,
        1/mass * (Frx - Ffy*jnp.sin(steer)) + vy*omega - g * pitch,
        1/mass * (Fry + Ffy*jnp.cos(steer)) - vx*omega + g * roll,
        1/Iz * (Ffy* lf*jnp.cos(steer) - Fry * lr)
    ])
    

@jax.jit
def _calc_forces(bank_params,  x, u):
    acc = u[0]
    steer = u[1]
    psi = x[2]
    vx = x[3]
    vy = x[4]
    omega = x[5]

    Bf, Br, Cf, Cr, Df, Dr, Cro, Cd, Ce, Cm = bank_params
    mass, Iz, lf, lr, pitch, roll =  3.74, 0.04712, 0.15875,0.17145, 0, 0

    Frx = mass * (acc * Ce - Cm * vx ) - Cro - Cd * (vx * vx)

    alphaf = steer - jnp.arctan2((lf*omega + vy), jnp.abs(vx))
    alphar =jnp.arctan2((lr*omega - vy), jnp.abs(vx))
    Ffy = Df * jnp.sin(Cf * jnp.arctan(Bf * alphaf))
    Fry = Dr * jnp.sin(Cr * jnp.arctan(Br * alphar))

    return Ffy, Frx, Fry # each of these should end up being num_models long
        
# @jitclass(spec)
class DBMPacejkaBank():
    def __init__(self, 
                 lf, lr, 
                 mass, Iz, 
                 Bf, Br,
                 Cf, Cr, 
                 Df, Dr, 
                 Cro, Cd,
                 Ce, Cm, 
                 num_models
                 ):
        # non-varying parameters
        self.lf = lf
        self.lr = lr
        self.mass = mass
        self.Iz = Iz

        # varying parameters
        self.Bf = Bf
        self.Br = Br
        self.Cf = Cf
        self.Cr = Cr
        self.Df = Df
        self.Dr = Dr
        self.Cro = Cro
        self.Cd = Cd
        self.Ce = Ce
        self.Cm = Cm

        # non-sampled state parameters
        self.roll = 0
        self.pitch = 0

        self.num_models = num_models

        self.param_bank = jnp.stack([
            self.Bf, self.Br, self.Cf, self.Cr, self.Df, self.Dr,
            self.Cro, self.Cd, self.Ce, self.Cm
        ], axis=1)


    def set_roll_pitch(self, roll, pitch): 
        # non-sampled state parameters (i.e. state parameters not differentiated)
        # are updated here, as they are not given to diffiequation via x_batch
        self.roll = roll
        self.pitch = pitch

    def get_known_params(self):
        return jnp.array([self.mass, self.Iz, self.lf, self.lr, self.pitch, self.roll])

    def get_bank_params(self):
        return self.param_bank

    def get_model_params_arr(self, index):
        return np.array([
            self.Bf[index],
            self.Br[index],
            self.Cf[index],
            self.Cr[index],
            self.Df[index],
            self.Dr[index],
            self.Cro[index],
            self.Cd[index],
            self.Ce[index],
            self.Cm[index]
        ])

    

    
    # def integrate_batch(self, x_t_batch, u_t_batch, t_start, t_end):
    #     """Batched version of _integrate"""

    #     odesol = self.odeintRK4_batch(x_t_batch, np.array([t_start, t_end]), u_t_batch)
    #     return odesol[-1]
    
    
    # def get_model_params(self, index):
    #     return {
    #         'Bf': self.Bf[index], 
    #         'Br': self.Br[index],
    #         'Cf': self.Cf[index], 
    #         'Cr': self.Cr[index], 
    #         'Df': self.Df[index], 
    #         'Dr': self.Dr[index], 
    #         'Cro':self.Cro[index], 
    #         'Cd': self.Cd[index], 
    #         'Ce': self.Ce[index], 
    #         'Cm': self.Cm[index], 
    #     }
  
    
    


In [3]:
import time
import jax
import jax.numpy as jnp
import numpy as np

# Assuming your integrator, diffequation, _calc_forces, DBMPacejkaBank,
# get_lookback_error, find_best_model, and LBHistory class
# are already defined/imported here

def benchmark_lbhistory_loop():
    num_models = 30000
    history_length = 20
    dt = 0.02
    state_size = 6
    cost_weights = jnp.ones(state_size)

    # Initialize model bank with dummy parameters (replace with your data)
    bank_params = jnp.array(np.random.rand(num_models, 10), dtype=jnp.float32)

    dynamics_bank = DBMPacejkaBank(
        lf=0.15875 * jnp.ones(num_models),
        lr=0.17145 * jnp.ones(num_models),
        mass=3.74 * jnp.ones(num_models),
        Iz=0.04712 * jnp.ones(num_models),
        Bf=bank_params[:, 0],
        Br=bank_params[:, 1],
        Cf=bank_params[:, 2],
        Cr=bank_params[:, 3],
        Df=bank_params[:, 4],
        Dr=bank_params[:, 5],
        Cro=bank_params[:, 6],
        Cd=bank_params[:, 7],
        Ce=bank_params[:, 8],
        Cm=bank_params[:, 9],
        num_models=num_models,
    )

    # Create LBHistory object
    lb_history = LBHistory(
        num_models=num_models,
        history_length=history_length,
        dt=dt,
        cost_weights=cost_weights,
        state_size=state_size,
        integrator=integrator,
        dynamics_bank=dynamics_bank,
        diffeq=diffequation,
    )

    # Initial state and control input
    x_t = jnp.zeros(state_size)
    u_t = jnp.array([1.0, 0.1])  # sample control input

    # Warm-up to trigger JIT compilation
    lb_history.predict_states(x_t, u_t)
    lb_history.update_lookback_error(x_t)

    # Benchmark loop
    n_steps = 5000
    predict_times = []

    for _ in range(n_steps):
        start = time.perf_counter()
        lb_history.predict_states(x_t, u_t)
        end = time.perf_counter()
        predict_times.append(end - start)

        lb_history.update_lookback_error(x_t)
        x_t = lb_history.last_predicted_states[0]

    avg_predict_time_ms = (sum(predict_times) / n_steps) * 1000
    print(f"Average predict_states time: {avg_predict_time_ms:.4f} ms")

if __name__ == "__main__":
    benchmark_lbhistory_loop()


Average predict_states time: 11.9257 ms
